### for cleaning and merging the importance question data responses: both from human and the llm variant

In [10]:
# cell1: inspecting human data
import pandas as pd
from pathlib import Path

df_human = pd.read_csv('raw-responses/importance_humanOnly_rq1.csv')

df_human.info()

# rename columns to match
df_human.rename(columns={'response_id': 'row_id'}, inplace=True)
df_human.rename(columns={'variant_id': 'model'}, inplace=True)
df_human.rename(columns={'respondent_id': 'variant_id'}, inplace=True)
df_human.rename(columns={'technology': 'dc_solution'}, inplace=True)
df_human.rename(columns={'rating_numeric': 'rating'}, inplace=True)
df_human.rename(columns={'rating_text': 'label'}, inplace=True)

# save new csv
df_human.to_csv('raw-responses/importance_humanOnly_rq1_renamed.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162 entries, 0 to 161
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   response_id     162 non-null    object 
 1   respondent_id   162 non-null    object 
 2   condition       162 non-null    object 
 3   variant_id      162 non-null    object 
 4   technology      162 non-null    object 
 5   rating_numeric  162 non-null    int64  
 6   rating_text     162 non-null    object 
 7   justification   0 non-null      float64
dtypes: float64(1), int64(1), object(6)
memory usage: 10.3+ KB


In [6]:
# cell2: inspecting llm variants data
df_llm = pd.read_csv('../1_zero-shot/zero-shot-raw/zeroshotImportanceResponses.csv')

df_llm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6600 entries, 0 to 6599
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   row_id         6600 non-null   object
 1   base_model     6600 non-null   object
 2   variant_id     6600 non-null   object
 3   model          6600 non-null   object
 4   dc_solution    6600 non-null   object
 5   rating         6600 non-null   int64 
 6   label          6600 non-null   object
 7   iteration      6600 non-null   int64 
 8   timestamp      6600 non-null   object
 9   justification  6600 non-null   object
dtypes: int64(2), object(8)
memory usage: 515.8+ KB


In [14]:
# cell3: merging human and llm data
import pandas as pd

# load data
df_human = pd.read_csv('raw-responses/importance_humanOnly_rq1_renamed.csv')
df_llm = pd.read_csv('../1_zero-shot/zero-shot-raw/zeroshotImportanceResponses.csv')

# create output directory
output_dir = Path('imp-zeroshot-working')
output_dir.mkdir(parents=True, exist_ok=True)

# define source and condition
df_human['source'] = 'human'
df_llm['source'] = 'llm'

df_human['condition'] = 'zeroshot'
df_llm['condition'] = 'zeroshot'

# define base model
df_human['base_model'] = 'human'

df_llm['base_model'] = (df_llm['variant_id'].astype('string').str.split('_', n=1).str[0])

# exact columns to keep
columns_to_keep = ['row_id', 'variant_id', 'base_model', 'source', 'model', 'dc_solution', 'rating', 'label', 'iteration', 'condition']


# add any missing columns
for column in columns_to_keep:
    if column not in df_human.columns: df_human[column] = pd.NA
    if column not in df_llm.columns: df_llm[column] = pd.NA

# combine the dataframes
df_combined = pd.concat([df_human[columns_to_keep], df_llm[columns_to_keep]], ignore_index=True)

# save combined dataframe
df_combined.to_csv(output_dir / 'importance-humanllm-responses.csv', index=False)


In [15]:
df_comb = pd.read_csv(output_dir / 'importance-humanllm-responses.csv')

df_comb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6762 entries, 0 to 6761
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   row_id       6762 non-null   object 
 1   variant_id   6762 non-null   object 
 2   base_model   6762 non-null   object 
 3   source       6762 non-null   object 
 4   model        6762 non-null   object 
 5   dc_solution  6762 non-null   object 
 6   rating       6762 non-null   int64  
 7   label        6762 non-null   object 
 8   iteration    6600 non-null   float64
 9   condition    6762 non-null   object 
dtypes: float64(1), int64(1), object(8)
memory usage: 528.4+ KB
